# 04 - IEEE-CIS LTN-Style Fraud Rule Analysis

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def find_project_root() -> Path | None:
    direct_candidates = [KAGGLE_PROJECT_DIR, Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = sorted(path.parent for path in base.glob("**/configs") if path.is_dir())
            for candidate in matches:
                if (candidate / "src").is_dir():
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and KAGGLE:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
        check=True,
    )
    PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Rules và quantile thresholds chỉ fit trên train. Notebook đánh giá rule truth values,
class-balanced knowledge-base satisfaction và một diagnostic fuzzy predicate khả vi.
Explicit numeric thresholds dùng softness theo đơn vị gốc; quantile/category-risk predicates
dùng relative softness. `transaction_hour` chỉ là cyclic phase suy ra từ TransactionDT,
không được diễn giải như giờ địa phương đã biết. Đây là LTN-style explanation layer,
không phải end-to-end LTN predictor.

In [ ]:
from src.artifacts import sha256_file, stable_config_hash
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data
from src.explanation import rule_quality_table
from src.logic import FraudKnowledgeBase, FraudRuleEngine

config_path = PROJECT_ROOT / "configs/ieee_cis.yaml"
config = load_config(config_path)
output_dir = OUTPUT_BASE / "04_ieee_cis_ltn_rule_analysis"
output_dir.mkdir(parents=True, exist_ok=True)
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
if not QUICK_RUN and str(data_source).lower() == "synthetic":
    raise ValueError("Full thesis rule analysis cannot consume synthetic-fallback data")
prepared = prepare_dataset(frame, config)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
print({"data_source": data_source, "active_rules": len(engine.rules), "skipped": engine.skipped_rules})
fitted_thresholds = engine.fitted_thresholds()
rationale_rows = []
for definition in config["logic"]["rules"]:
    operators = [condition["operator"] for condition in definition["conditions"]]
    has_category_risk = "category_risk" in operators
    has_quantile = any(operator.endswith("_quantile") for operator in operators)
    if has_category_risk and has_quantile:
        knowledge_type = "data-informed and train-fitted fuzzy hypothesis"
        threshold_source = (
            "train-label-smoothed category risk with configured cutoff plus training-split quantile"
        )
    elif has_category_risk:
        knowledge_type = "data-informed fuzzy hypothesis"
        threshold_source = "train-label-smoothed category risk with configured cutoff"
    elif has_quantile:
        knowledge_type = "domain hypothesis with train-fitted thresholds"
        threshold_source = "training-split quantile"
    else:
        knowledge_type = "configured domain hypothesis"
        threshold_source = "explicit configured value"
    limitation = (
        "TransactionDT-derived cyclic phase; the unknown clock origin prevents a local-hour claim."
        if definition["name"] == "unusual_transaction_hour"
        else "Association-based audit evidence; activation does not establish causality or model faithfulness."
    )
    rationale_rows.append({
        "rule": definition["name"],
        "description": definition.get("description", ""),
        "knowledge_type": knowledge_type,
        "threshold_source": threshold_source,
        "limitation": limitation,
    })
rule_rationale = pd.DataFrame(rationale_rows)
display(fitted_thresholds, rule_rationale)
rule_rationale.to_csv(output_dir / "ieee_rule_rationale.csv", index=False)

## Differentiable fuzzy predicate diagnostic

In [ ]:
import torch
from src.logic import SoftThresholdPredicate, TensorLogic

amount = torch.tensor(prepared.train_frame["TransactionAmt"].to_numpy(float), dtype=torch.float32)
center = torch.nanmedian(amount)
scale = torch.nan_to_num(amount.std(), nan=1.0).clamp_min(1e-6)
values = torch.nan_to_num((amount - center) / scale)
labels = torch.tensor(prepared.y_train, dtype=torch.float32)
predicate = SoftThresholdPredicate(float(torch.quantile(values, 0.90)), temperature=0.5, learnable=True)
optimizer = torch.optim.Adam(predicate.parameters(), lr=0.03)
initial_threshold = float(predicate.threshold.detach())
history = []
for _ in range(30 if QUICK_RUN else 100):
    optimizer.zero_grad()
    evidence = predicate(values)
    satisfaction = 0.5 * (
        TensorLogic.forall(evidence[labels == 1]) +
        TensorLogic.forall(1.0 - evidence[labels == 0])
    )
    loss = 1.0 - satisfaction + 0.01 * (predicate.threshold - initial_threshold).pow(2)
    loss.backward()
    optimizer.step()
    history.append(float(satisfaction.detach()))
tensor_demo = pd.DataFrame([{
    "initial_threshold": initial_threshold,
    "learned_threshold": float(predicate.threshold.detach()),
    "initial_satisfaction": history[0],
    "final_satisfaction": history[-1],
}])
display(tensor_demo.round(5))

## Rule and knowledge-base results

In [ ]:
split_frames = {
    "train": (prepared.train_frame, prepared.y_train),
    "validation": (prepared.validation_frame, prepared.y_validation),
    "test": (prepared.test_frame, prepared.y_test),
}
activation = float(config["logic"]["activation_threshold"])
quality_frames = []
satisfaction_rows = []
for split, (split_frame, labels) in split_frames.items():
    truth = engine.evaluate(split_frame)
    quality_frames.append(rule_quality_table(truth, labels, activation).assign(split=split))
    satisfaction_rows.append({"split": split, **knowledge_base.satisfaction_breakdown(split_frame, target)})
quality = pd.concat(quality_frames, ignore_index=True)
satisfaction = pd.DataFrame(satisfaction_rows)
display(quality.round(4), satisfaction.round(4))
quality.to_csv(output_dir / "ieee_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "ieee_knowledge_base_satisfaction.csv", index=False)
fitted_thresholds.to_csv(output_dir / "ieee_fitted_rule_thresholds.csv", index=False)
tensor_demo.to_csv(output_dir / "ieee_tensor_predicate_diagnostic.csv", index=False)

In [ ]:
validation_quality = quality.query("split == 'validation'")
test_quality = quality.query("split == 'test'")
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_delta", "lift_delta"]].round(4))
stability.to_csv(output_dir / "ieee_rule_stability.csv", index=False)
rule_output_files = [
    "ieee_rule_rationale.csv",
    "ieee_rule_quality.csv",
    "ieee_knowledge_base_satisfaction.csv",
    "ieee_fitted_rule_thresholds.csv",
    "ieee_tensor_predicate_diagnostic.csv",
    "ieee_rule_stability.csv",
]
rule_lineage = {
    "notebook_id": "04_IEEE_CIS_LTN_Rule_Analysis",
    "git_commit": GIT_COMMIT,
    "dataset_name": config["dataset"]["name"],
    "data_source": data_source,
    "quick_run": QUICK_RUN,
    "config_sha256": stable_config_hash(config),
    "audit_source_sha256": audit_pipeline_fingerprint(config_path),
    "output_files": rule_output_files,
    "output_sha256": {
        name: sha256_file(output_dir / name) for name in rule_output_files
    },
}
lineage_path = output_dir / "upstream_lineage.json"
lineage_path.write_text(json.dumps(rule_lineage, indent=2), encoding="utf-8")
print({"upstream_lineage": str(lineage_path)})

## Takeaways

In [ ]:
eligible_rules = test_quality.dropna(subset=["lift"]).query("active_count > 0")
strongest = eligible_rules.sort_values("lift", ascending=False).iloc[0]
test_satisfaction = satisfaction.query("split == 'test'").iloc[0]
display(Markdown(
    f"- Highest observed test lift: **{strongest['rule']} = {strongest['lift']:.3f}** "
    f"with **{int(strongest['active_count'])}** active rows; lift is never interpreted without this denominator.\n"
    f"- Test balanced knowledge-base satisfaction: **{test_satisfaction['balanced_satisfaction']:.3f}**.\n"
    "- Rule lift and satisfaction measure limited association/logic agreement, not causality or predictor faithfulness."
))